# Lección 04 — Mini Dashboard

Un dashboard es un panel que muestra información importante de un vistazo.

Vamos a construir uno para explorar videojuegos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

%matplotlib inline
pd.set_option("display.max_rows", 20)

df = pd.read_csv("../../../shared/data/videojuegos.csv")
print("✅ Datos cargados.")

## Definir los controles

In [ ]:
generos = ["Todos"] + sorted(df["genero"].unique().tolist())
metricas = ["calificacion", "horas_promedio"]

ctrl_genero = widgets.Dropdown(
    options=generos, value="Todos", description="Género:"
)

ctrl_metrica = widgets.Dropdown(
    options=metricas, value="calificacion", description="Métrica:"
)

ctrl_top_n = widgets.IntSlider(
    min=3, max=20, step=1, value=10, description="Top N:"
)

print("Controles listos.")

## La función del dashboard

In [ ]:
def mostrar_dashboard(genero, metrica, top_n):
    # Filtrar datos
    if genero == "Todos":
        datos = df.copy()
    else:
        datos = df[df["genero"] == genero].copy()
    
    datos = datos.sort_values(metrica, ascending=False).head(top_n)
    
    # Crear figura con dos paneles
    fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                             gridspec_kw={"width_ratios": [1, 2]})
    fig.suptitle(f"Dashboard de Videojuegos — {genero}", fontsize=14, fontweight="bold")

    # Panel izquierdo: métricas clave
    ax_info = axes[0]
    ax_info.axis("off")

    resumen = [
        ("Total juegos filtrados", len(datos)),
        (f"Promedio {metrica}", f"{datos[metrica].mean():.2f}"),
        (f"Máximo {metrica}", f"{datos[metrica].max():.2f}"),
        (f"Mínimo {metrica}", f"{datos[metrica].min():.2f}"),
        ("Mejor juego", datos.iloc[0]["titulo"] if len(datos) > 0 else "—"),
    ]

    y = 0.9
    for etiqueta, valor in resumen:
        ax_info.text(0.05, y, etiqueta, fontsize=10, color="gray", transform=ax_info.transAxes)
        ax_info.text(0.05, y - 0.06, str(valor), fontsize=13, fontweight="bold",
                    color="#2c3e50", transform=ax_info.transAxes)
        y -= 0.18

    # Panel derecho: gráfico de barras
    ax_graf = axes[1]
    colores = plt.cm.coolwarm([i / len(datos) for i in range(len(datos))])
    ax_graf.barh(datos["titulo"][::-1], datos[metrica][::-1], color=colores)
    ax_graf.set_title(f"Top {top_n} por {metrica}")
    ax_graf.set_xlabel(metrica)

    if metrica == "calificacion":
        ax_graf.set_xlim(7, 10)

    plt.tight_layout()
    plt.show()

print("Función del dashboard lista.")

## ¡Activa el dashboard!

In [ ]:
salida = widgets.interactive_output(
    mostrar_dashboard,
    {"genero": ctrl_genero, "metrica": ctrl_metrica, "top_n": ctrl_top_n}
)

panel_controles = widgets.VBox([
    widgets.HTML("<b>Filtros</b>"),
    ctrl_genero,
    ctrl_metrica,
    ctrl_top_n
], layout=widgets.Layout(width="220px", padding="10px"))

display(widgets.HBox([panel_controles, salida]))

## 🎯 Mini Reto

Modifica el panel izquierdo para que también muestre:
- El juego con **menor** calificación del filtro actual
- El año de lanzamiento del juego mejor calificado

Pista: accede a la última fila del dataframe ordenado con `datos.iloc[-1]`.

In [ ]:
# Tu versión mejorada aquí
